# 1. Imports & Config

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = 'data/accepted_2007_to_2018Q4.csv'
TRAIN_CUTOFF_YEAR = 2015
RANDOM_STATE = 42

# 2. Load Raw Data

In [2]:
import os

os.makedirs('data', exist_ok=True)

os.system('kaggle datasets download -d wordsforthewise/lending-club -p data/ --unzip')
df_raw = pd.read_csv(
    DATA_PATH,
    low_memory=False,
    skipfooter=2,
    engine='python'
)

print(df_raw.shape)
print(df_raw['loan_status'].value_counts())

Dataset URL: https://www.kaggle.com/datasets/wordsforthewise/lending-club
License(s): CC0-1.0


100%|█████████████████████████████████████| 1.26G/1.26G [01:53<00:00, 12.0MB/s]


ValueError: The 'low_memory' option is not supported with the 'python' engine

# 3. Filter to Usable Loan Statuses

In [ ]:
# Event statuses: loan defaulted
EVENT_STATUSES = {'Charged Off', 'Default'}

# Censored statuses: loan ended without default
CENSORED_STATUSES = {
    'Fully Paid',
    'Current',
    'In Grace Period',
    'Late (16-30 days)',
    'Late (31-120 days)'
}

keep = EVENT_STATUSES | CENSORED_STATUSES
df = df_raw[df_raw['loan_status'].isin(keep)].copy()

print(f"Rows after status filter: {len(df)}")
print(df['loan_status'].value_counts())

# 4. Construct Survival Time & Event Indicator

In [ ]:
df['issue_d'] = pd.to_datetime(df['issue_d'], format='%b-%Y')
df['last_pymnt_d'] = pd.to_datetime(df['last_pymnt_d'], format='%b-%Y')

# Drop rows where we can't compute duration
df = df.dropna(subset=['issue_d', 'last_pymnt_d'])

# Duration in months (floor)
df['duration'] = (
    (df['last_pymnt_d'].dt.year - df['issue_d'].dt.year) * 12
    + (df['last_pymnt_d'].dt.month - df['issue_d'].dt.month)
)

# Clamp: duration must be at least 1 month
df['duration'] = df['duration'].clip(lower=1)

df['event'] = df['loan_status'].isin(EVENT_STATUSES).astype(int)

print(df[['duration', 'event']].describe())
print(f"\nEvent rate: {df['event'].mean():.3f}")

# 5. Select Features (Origination-Time Only)

In [ ]:
# Only columns known at loan origination — nothing updated post-issuance
FEATURE_COLS = [
    'loan_amnt',
    'int_rate',
    'installment',
    'grade',
    'sub_grade',
    'emp_length',
    'home_ownership',
    'annual_inc',
    'verification_status',
    'purpose',
    'dti',
    'delinq_2yrs',
    'fico_range_low',
    'fico_range_high',
    'open_acc',
    'pub_rec',
    'revol_bal',
    'revol_util',
    'total_acc',
    'term',
]

TARGET_COLS = ['duration', 'event']
META_COLS = ['issue_d']

df = df[FEATURE_COLS + TARGET_COLS + META_COLS].copy()
print(df.shape)
print(df.isnull().sum().sort_values(ascending=False).head(15))

# 6. Clean Individual Columns

In [ ]:
# term: '36 months' -> 36
df['term'] = df['term'].str.extract(r'(\d+)').astype(float)

# int_rate: '13.5%' -> 13.5 (sometimes already float)
if df['int_rate'].dtype == object:
    df['int_rate'] = df['int_rate'].str.replace('%', '').astype(float)

# revol_util: same pattern
if df['revol_util'].dtype == object:
    df['revol_util'] = df['revol_util'].str.replace('%', '').astype(float)

# emp_length: '10+ years' -> 10, '< 1 year' -> 0
emp_map = {
    '< 1 year': 0,
    '1 year': 1, '2 years': 2, '3 years': 3, '4 years': 4,
    '5 years': 5, '6 years': 6, '7 years': 7, '8 years': 8,
    '9 years': 9, '10+ years': 10
}
df['emp_length'] = df['emp_length'].map(emp_map)

# home_ownership: collapse NONE/ANY into OTHER
df['home_ownership'] = df['home_ownership'].replace({'NONE': 'OTHER', 'ANY': 'OTHER'})

print(df.dtypes)

# 7. Imputation

In [ ]:
numeric_cols = df[FEATURE_COLS].select_dtypes(include='number').columns.tolist()
categorical_cols = df[FEATURE_COLS].select_dtypes(include='object').columns.tolist()

# Median for numeric
for col in numeric_cols:
    median = df[col].median()
    df[col] = df[col].fillna(median)

# Mode for categorical
for col in categorical_cols:
    mode = df[col].mode()[0]
    df[col] = df[col].fillna(mode)

print(f"Nulls remaining: {df.isnull().sum().sum()}")

# 8. Encoding

In [ ]:
# Ordinal: grade and sub_grade have a natural risk ordering
grade_order = ['A', 'B', 'C', 'D', 'E', 'F', 'G']
grade_map = {g: i+1 for i, g in enumerate(grade_order)}
df['grade'] = df['grade'].map(grade_map)

subgrade_order = [f"{g}{n}" for g in grade_order for n in range(1, 6)]
subgrade_map = {sg: i+1 for i, sg in enumerate(subgrade_order)}
df['sub_grade'] = df['sub_grade'].map(subgrade_map)

# One-hot: no natural ordering
ohe_cols = ['home_ownership', 'purpose', 'verification_status']
df = pd.get_dummies(df, columns=ohe_cols, drop_first=True)

print(df.shape)
print(df.dtypes.value_counts())

# 9. Temporal Train / Val / Test Split

In [ ]:
# Train: issued before cutoff
# Val: cutoff year
# Test: after cutoff year
train_mask = df['issue_d'].dt.year < TRAIN_CUTOFF_YEAR
val_mask   = df['issue_d'].dt.year == TRAIN_CUTOFF_YEAR
test_mask  = df['issue_d'].dt.year > TRAIN_CUTOFF_YEAR

train = df[train_mask].drop(columns=['issue_d'])
val   = df[val_mask].drop(columns=['issue_d'])
test  = df[test_mask].drop(columns=['issue_d'])

print(f"Train: {len(train)} | Val: {len(val)} | Test: {len(test)}")
print(f"Event rates  -> train: {train['event'].mean():.3f} | val: {val['event'].mean():.3f} | test: {test['event'].mean():.3f}")

# 10. Save Processed Splits

In [ ]:
import os
os.makedirs('data/processed', exist_ok=True)

train.to_parquet('data/processed/train.parquet', index=False)
val.to_parquet('data/processed/val.parquet', index=False)
test.to_parquet('data/processed/test.parquet', index=False)

print('saved to data/processed/')

# 11. Sanity Checks

In [ ]:
# Duration distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

train['duration'].hist(bins=50, ax=axes[0])
axes[0].set_title('duration distribution (train)')
axes[0].set_xlabel('months')

train.groupby('event')['duration'].hist(bins=50, alpha=0.6, ax=axes[1])
axes[1].set_title('duration by event (train)')
axes[1].set_xlabel('months')
axes[1].legend(['censored (0)', 'event (1)'])

plt.tight_layout()
plt.show()

In [ ]:
# Make sure no leakage columns slipped through
leakage_keywords = ['recoveries', 'total_pymnt', 'out_prncp', 'collection', 'last_pymnt']
found = [c for c in train.columns if any(kw in c for kw in leakage_keywords)]
print(f"Potential leakage columns: {found}")
assert len(found) == 0, 'remove these before modeling'

# Duration sanity
assert (train['duration'] >= 1).all()
assert train['event'].isin([0, 1]).all()

print('all checks passed')